# Comparison with observations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

import utilities.plot_settings

Compare the distributions of simulated pulsars with neutron stars in the ATNF catalogue. We select from the ATNF catalog all neutron stars that are presumably isolated and not recycled. We also compare with the sub-sample of those that have a measured proper motion.

In [ ]:
# Read full atnf catalog (binaries excluded) .csv file.
data_atnf = pd.read_csv("../examples/data/atnf_full_nobinary_13-11-2020.csv", delimiter=',', header=[0,1])
data_atnf.head()

In [ ]:
# Select only stars with measure of P and Pdot and that are not in globular clusters or in the Magellanic Clouds
data_atnf = data_atnf[~data_atnf["P0"]["[s]"].isin(['NAN'])]
data_atnf = data_atnf[~data_atnf["P1"]["[s/s]"].isin(['NAN'])]
data_atnf = data_atnf[~data_atnf["DIST"]["[kpc]"].isin(['NAN'])]
data_atnf = data_atnf[~data_atnf["ASSOC"]["Unnamed: 21_level_1"].isin(['EXGAL:SMC', 'EXGAL:LMC', 'GC:47Tuc', 'GC:M3', 'GC:M5', 'GC:M13', 'GC:NGC6440', 'GC:Ter5', 'GC:NGC6441', 'GC:NGC6517', 'GC:NGC6522', 'GC:NGC6624', 'GC:M28(NGC6626)', 'GC:NGC6652', 'GC:M22(NGC6656)', 'GC:NGC6752', 'GC:NGC6760', 'GC:M15', 'GC:M30'])]

RA_atnf = data_atnf["RAJD"]["[deg]"].to_numpy().astype(np.float) 
DEC_atnf = data_atnf["DECJD"]["[deg]"].to_numpy().astype(np.float) 
P_atnf = data_atnf["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_atnf = data_atnf["P1"]["[s/s]"].to_numpy().astype(np.float) 
dist_atnf = data_atnf["DIST"]["[kpc]"].to_numpy().astype(np.float)

# select only isolated non recycled neutron stars i.e. with Pdot > 1e-17 and with an estimate of the distance  from the DM which is < 25 kpc
cond = (Pdot_atnf > 1e-17) & (dist_atnf < 25)
RA_atnf = RA_atnf[cond]
DEC_atnf = DEC_atnf[cond]
dist_atnf = dist_atnf[cond]

In [ ]:
# Read observed proper motion neutron stars .csv file
data_pm = pd.read_csv("../examples/data/PSRs_prop_motion_22-05-2020.csv", header=[0,1])
data_pm.head()

# Select only stars with measure of P and Pdot and that are not in globular clusters
data_pm = data_pm[~data_pm["P0"]["[s]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["P1"]["[s/s]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["DIST_DM"]["[kpc]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["ASSOC"]["Unnamed: 24_level_1"].isin(['EXGAL:SMC', 'EXGAL:LMC', 'GC:47Tuc', 'GC:M3', 'GC:M5', 'GC:M13', 'GC:NGC6440', 'GC:Ter5', 'GC:NGC6441', 'GC:NGC6517', 'GC:NGC6522', 'GC:NGC6624', 'GC:M28(NGC6626)', 'GC:NGC6652', 'GC:M22(NGC6656)', 'GC:NGC6752', 'GC:NGC6760', 'GC:M15', 'GC:M30'])]

# Extract parameters
RA_pm = data_pm["RAJD"]["[deg]"].to_numpy().astype(np.float) 
DEC_pm = data_pm["DECJD"]["[deg]"].to_numpy().astype(np.float) 
pmRA_pm = data_pm["PMRA"]["[mas/yr]"].to_numpy().astype(np.float) 
pmRA_err_pm = data_pm["PMRA_err"]["[mas/yr]"].to_numpy().astype(np.float) 
pmDEC_pm = data_pm["PMDEC"]["[mas/yr]"].to_numpy().astype(np.float) 
pmDEC_err_pm = data_pm["PMDEC_err"]["[mas/yr]"].to_numpy().astype(np.float) 
dist_pm = data_pm["DIST_DM"]["[kpc]"].to_numpy().astype(np.float) 
NS_class = data_pm["CLASS"]["Unnamed: 13_level_1"].to_numpy()
P_pm = data_pm["P0"]["[s]"].to_numpy().astype(np.float) 
Pdot_pm = data_pm["P1"]["[s/s]"].to_numpy().astype(np.float) 
assoc = data_pm["ASSOC"]["Unnamed: 24_level_1"].to_numpy()

# select only isolated non recycled neutron stars i.e. with Pdot > 1e-17 and distance < 25 kpc
cond = (Pdot_pm > 1e-17) & (NS_class != "Binary PSR")  & (dist_pm < 25)
RA_pm = RA_pm[cond]
DEC_pm = DEC_pm[cond]
pmRA_pm = pmRA_pm[cond]
pmRA_err_pm = pmRA_err_pm[cond]
pmDEC_pm = pmDEC_pm[cond]
pmDEC_err_pm = pmDEC_err_pm[cond]
dist_pm = dist_pm[cond]

In [ ]:
# Select a `final_population.pkl.gz` file to import:
data = pd.read_pickle("../examples/data/simulation_maxwell_sigma265_h018/final_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
RA = data["RA"]["[deg]"].to_numpy()
DEC = data["DEC"]["[deg]"].to_numpy()
pmRA = data["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC = data["pm_DEC"]["[mas yr^-1]"].to_numpy()
v_r = data["v_r"]["[km s^-1]"].to_numpy()
v_phi = data["v_phi"]["[km s^-1]"].to_numpy()
v_z = data["v_z"]["[km s^-1]"].to_numpy()
d = data["d"]["[kpc]"].to_numpy()

Introduce some selection biases as a function of the distance

In [ ]:
def calculate_selection_weights(d: np.ndarray) -> np.ndarray:
    """
    Calculate the weights to assign to every star for selection.
    Weights are evaluated as a function of distance, nearest stars are easier and more likely to be detected

    Args:
        d (np.ndarray): array of distances from the Sun [kpc].

    Returns:
        (np.ndarray): array of selection weights.
    """

    # This function has been fine tuned to match the distance distribution of
    # the neutron stars with observed proper motion.
    weights = np.exp(-0.5 * d) / d

    # Normalize the weights to their sum
    w = weights / np.sum(weights)

    return w

# calculate selection weights that are function of the distance.
w = calculate_selection_weights(d)

Select a number of simulated NSs equal to the one in the proper motion observed sample according to the selection function above. Then compare the distribution of distances of the observed neutron stars and the resampled simulated population with a K-S (Kolmogorov-Smirnov) test. The K-S statistic is averaged over 1000 comparisons.



In [ ]:
# Select a number of simulated NSs equal to the one in the proper motion observed sample according to the selection function above.
data_select = data.sample(len(RA_pm), replace=False, weights=w)
d_sel = data_select["d"]["[kpc]"].to_numpy()
  
RA_sel = data_select["RA"]["[deg]"].to_numpy()
DEC_sel = data_select["DEC"]["[deg]"].to_numpy()
pmRA_sel = data_select["pm_RA"]["[mas yr^-1]"].to_numpy()
pmDEC_sel = data_select["pm_DEC"]["[mas yr^-1]"].to_numpy()
vls_sel = data_select["v_ls"]["[km s^-1]"].to_numpy()
d_sel = data_select["d"]["[kpc]"].to_numpy()

# compare the distribution of the observed and simulated selected sample with K-S test and average statistics over n_trials
n_trials = 1000
p_values = np.zeros(n_trials)
for i in range(n_trials):
    df_select = data.sample(len(RA_pm), replace=False, weights=w)
    d_sel_test = df_select["d"]["[kpc]"].to_numpy()
    # perform the K-S test on the observed and simulated samples
    stat, p = stats.ks_2samp(dist_pm, d_sel_test)
    p_values[i] = p
    
print(np.mean(p_values))

Histogramming the distance from the Sun.

In [ ]:
d_edges = np.linspace(0, 25, 31)

fig, ax = plt.subplots()

ax.hist(
    dist_atnf,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="tab:green",
    facecolor="tab:green",
    lw=4,
    alpha=1.,
    label=r"Observed full atnf",
    rasterized=True
)
ax.hist(
    dist_pm,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1.,
    label=r"Observed proper motion",
    rasterized=True
)
ax.hist(
    d,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.,
    label=r"Simulated all",
    rasterized=True
)
ax.hist(
    d_sel,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.,
    label=r"Simulated + bias",
    zorder=5,
    rasterized=True
)
ax.set_xlabel(r'$d_{\odot}$ [kpc]')
ax.set_ylabel('Number of NSs')
ax.set_yscale('log')
ax.set_xlim(0.,25.)
ax.legend(frameon=False, loc=2, fontsize=24)

plt.show(block=False)

Plotting the distribution in RA and DEC in the ICRS (International Celestial Reference System).

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0

fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    RA, 
    DEC, 
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=3,
    alpha=0.3,
    rasterized=True,
    label=r"Simulated all",
)
ax.plot(
    RA_atnf, 
    DEC_atnf, 
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed full ATNF",
)
ax.plot(
    RA_pm, 
    DEC_pm, 
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Observed proper motion",
)
ax.plot(
    RA_sel, 
    DEC_sel, 
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=3,
    alpha=1,
    rasterized=True,
    label=r"Simulated + bias",
)


ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:red", markersize=20)
ax.set_xlim(0., 360.)
ax.set_ylim(-90., 90.)
ax.set_xlabel('RA [deg]')
ax.set_ylabel('DEC [deg]')
ax.legend(frameon=True, loc=0)

plt.show()

Histrogramming RA and DEC coordinate position.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0., 360., 61)

ax.hist(
    RA_atnf,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="tab:green",
    facecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Observed full ATNF",
)
ax.hist(
    RA_pm,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    RA,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    RA_sel,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale('log')
ax.set_xlim(0., 360.)
ax.legend(frameon=False, loc=2)

plt.show()

In [ ]:
fig, ax = plt.subplots()

DEC_edges = np.linspace(-90., 90., 31)

ax.hist(
    DEC_atnf,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="tab:green",
    facecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Observed full ATNF",
)
ax.hist(
    DEC_pm,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    DEC,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    DEC_sel,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale('log')
ax.set_xlim(-90., 90.)
ax.legend(frameon=False, loc=2)

plt.show()

Histrogramming the angular proper velocity in RA and DEC.

In [ ]:
fig, ax = plt.subplots()

pm_edges = np.linspace(-200, 200, 31)

ax.hist(
    pmRA_pm,
    bins=pm_edges,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    pmRA,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    pmRA_sel,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [mas yr$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale('log')
ax.legend(frameon=False, loc=2)

plt.show()

In [ ]:
ig, ax = plt.subplots()

ax.hist(
    pmDEC_pm,
    bins=pm_edges,
    histtype="stepfilled",
    edgecolor="tab:orange",
    facecolor="tab:orange",
    lw=4,
    alpha=1,
    label=r"Observed proper motion",
)
ax.hist(
    pmDEC,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    pmDEC_sel,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated + bias",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [mas yr$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
ax.legend(frameon=False, loc=2)
ax.set_yscale('log')

plt.show()